In [ ]:
include( "../model.jl" )
include( "../prob.jl" )

using ThreadsX
using StatsBase
using Printf

using StatsPlots
using Distributions

home, base = expfolder( "system" )

In [ ]:
# System parameters.
N = 1024

# Import noise, speed, and density lists.
include( "../0-params/nos-list.jl" )
include( "../0-params/rho-list.jl" )

# Color map.
# colormap = cgrad( [:black, :gray, :white, :mediumpurple], [0.0, 0.10, 0.40, 0.50, 0.75, 1.0] )
colormap = cgrad( :ice )

In [ ]:
# Initialize folder names.
s = 0.1
folderdata = [findfolder( N, ρlist[i], s, ηlist[k]; base=base ) for i ∈ 1:Nρ, k ∈ 1:Nη];

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ, "Extracting critical noise information." )

# Initialize mean order and critical noise matrix.
μφdata = Matrix{defFloat}( undef, Nρ, Nη )

# Iterate through the data.
T = 10_000
@threads for i ∈ 1:Nρ
    # Import order parameter data.
    φlist = hcat( [vec( readdlm( folderdata[i,k]*"order_$(T)-end.txt" ) ) for k ∈ 1:Nη]... )

    # Compute the mean order parameter.
    μφdata[i,:] = vec( mean( φlist; dims=1 ) )

    # Update progress meter.
    next!( meter )
end

In [ ]:
# Extract the critical noise for each (ρ,s) pair.
ηclist = [criticalnoise( ηlist, μφdata[i,:] ) for i ∈ 1:Nρ];

In [ ]:
# Plot an example order-disorder trend.
plt = plot(; size=(400,250), legend=false )

for i ∈ 1:Nρ
    plot!( plt, ηlist, μφdata[i,:]; color=colormap[(i-1)/(Nρ-1)], marker=:circ, lw=2 )
end

plot!( plt; xlims=(0,ηlist[end]), ylims=(0,1) )
plot!( plt; xlabel="noise amplitude, "*L"η", ylabel="mean order, "*L"⟨φ⟩" )

In [ ]:
# Save data to pass to the HPC.
if false
    for i ∈ 1:Nρ
        savefolder = folderdata[i,1]*"../"
        writedlm( savefolder*"order-0.5.txt", ηclist[i] )
    end
end